# 01 — General Dataset Exploration

This notebook provides a **complete, professional EDA** workflow you can reuse:
- Load a dataset (CSV/DataFrame)
- Inspect schema, dtypes, missingness, and cardinality
- Plot distributions per feature (numeric & categorical)
- Detect and optionally remove outliers (IQR rule)
- Compute correlations (Pearson for numeric; Cramér's V for categorical)
- Analyze **TARGET_COLUMN** (regression/classification heuristics)
- Save a cleaned CSV to `artifacts/cleaned.csv`

> Plots are plain **matplotlib** (one chart per figure), no custom styles or colors for portability.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional

from scipy import stats
from scipy.stats import chi2_contingency
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    mean_squared_error, r2_score, mean_absolute_error
)

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# === Load your dataset ==========================================================
CSV_PATH = "your_dataset.csv"   # e.g., "../data/your_file.csv"
TARGET_COLUMN = None            # e.g., "price" or "target"; set to None if unknown

if not Path(CSV_PATH).exists():
    rng = np.random.default_rng(42)
    n = 800
    df = pd.DataFrame({
        "age": rng.normal(40, 12, n).round(0),
        "income": rng.lognormal(mean=10, sigma=0.5, size=n),
        "city": rng.choice(["Rome", "Milan", "Naples", "Turin"], size=n, replace=True),
        "is_premium": rng.choice([0,1], size=n, replace=True, p=[0.7,0.3]),
    })
    df["spend"] = 200 + 0.8*df["income"] + 3.0*df["age"] + rng.normal(0, 200, n)
    TARGET_COLUMN = "spend"
    print("Loaded synthetic dataset — set CSV_PATH to your file to use a real dataset.")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded: {CSV_PATH}")

print(df.shape)
df.head(3)

In [ ]:
def split_feature_types(df: pd.DataFrame, target: Optional[str]=None) -> Tuple[List[str], List[str]]:
    cats, nums = [], []
    for c in df.columns:
        if target is not None and c == target:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            nums.append(c)
        else:
            cats.append(c)
    return cats, nums

def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    m = df.isna().sum().sort_values(ascending=False)
    pct = (m / len(df)).round(4)
    out = pd.DataFrame({"missing": m, "missing_pct": pct})
    return out[out["missing"] > 0]

def cardinality(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    return pd.DataFrame({"unique": [df[c].nunique() for c in cols]}, index=cols).sort_values("unique", ascending=False)

def iqr_outlier_mask(s: pd.Series, k: float = 1.5) -> pd.Series:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k*iqr, q3 + k*iqr
    return (s < lo) | (s > hi)

def cramers_v(x: pd.Series, y: pd.Series) -> float:
    tbl = pd.crosstab(x, y)
    chi2, p, dof, ex = chi2_contingency(tbl, correction=False)
    n = tbl.values.sum()
    return float(np.sqrt((chi2 / n) / (min(tbl.shape) - 1)))

In [ ]:
print("Rows, Columns:", df.shape)
print("\nDtypes:\n", df.dtypes)

cats, nums = split_feature_types(df, target=TARGET_COLUMN)
print("\nCategorical features:", cats)
print("Numeric features:", nums)

print("\nMissing values summary:")
ms = missing_summary(df)
display(ms if not ms.empty else pd.DataFrame({"info":["No missing values detected."]}))

print("\nCategorical cardinality:")
display(cardinality(df, cats) if cats else pd.DataFrame({"info":["No categorical features detected."]}))

In [ ]:
MAX_PLOTS = 20
for i, col in enumerate(nums[:MAX_PLOTS], start=1):
    plt.figure()
    df[col].dropna().plot(kind="hist", bins=30, alpha=1.0)
    plt.title(f"Histogram — {col}"); plt.xlabel(col); plt.ylabel("Count")
    plt.show()

    plt.figure()
    plt.boxplot(df[col].dropna(), vert=True, labels=[col])
    plt.title(f"Boxplot — {col}")
    plt.show()

In [ ]:
for i, col in enumerate(cats[:MAX_PLOTS], start=1):
    vc = df[col].astype("object").value_counts(dropna=False).head(25)
    plt.figure()
    plt.bar(vc.index.astype(str), vc.values)
    plt.title(f"Top categories — {col}"); plt.xticks(rotation=45, ha="right"); plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
if nums:
    masks = {c: iqr_outlier_mask(df[c].astype(float)) for c in nums}
    combined = np.zeros(len(df), dtype=bool)
    for m in masks.values():
        combined |= m.fillna(False).values
    outlier_count = int(combined.sum())
    print(f"Potential outliers detected (IQR rule across any numeric col): {outlier_count}")
else:
    combined = np.zeros(len(df), dtype=bool)
    print("No numeric columns for outlier detection.")

REMOVE_OUTLIERS = True
df_no_outliers = df.loc[~combined].copy() if REMOVE_OUTLIERS else df.copy()
print("After outlier removal:", df_no_outliers.shape)

In [ ]:
if len(nums) >= 2:
    corr = df_no_outliers[nums].corr(numeric_only=True).fillna(0.0)
    plt.figure(figsize=(6,5))
    plt.imshow(corr.values, interpolation="nearest")
    plt.xticks(range(len(nums)), nums, rotation=45, ha="right")
    plt.yticks(range(len(nums)), nums)
    plt.title("Pearson Correlation (numeric)"); plt.colorbar()
    plt.tight_layout(); plt.show()
else:
    print("Not enough numeric features for correlation heatmap.")

if len(cats) >= 2:
    cv_mat = np.zeros((len(cats), len(cats)))
    for i, ci in enumerate(cats):
        for j, cj in enumerate(cats):
            if i == j:
                cv_mat[i, j] = 1.0
            elif i < j:
                try:
                    cv = cramers_v(df_no_outliers[ci].astype("object"), df_no_outliers[cj].astype("object"))
                except Exception:
                    cv = np.nan
                cv_mat[i, j] = cv_mat[j, i] = cv
    plt.figure(figsize=(6,5))
    plt.imshow(cv_mat, interpolation="nearest")
    plt.xticks(range(len(cats)), cats, rotation=45, ha="right")
    plt.yticks(range(len(cats)), cats)
    plt.title("Cramér's V (categorical)"); plt.colorbar()
    plt.tight_layout(); plt.show()
else:
    print("Not enough categorical features for Cramér's V heatmap.")

In [ ]:
TOP_K_PAIRS = 5
if len(nums) >= 2:
    corr = df_no_outliers[nums].corr(numeric_only=True).abs()
    pairs = []
    for i in range(len(nums)):
        for j in range(i+1, len(nums)):
            pairs.append(((nums[i], nums[j]), corr.iloc[i, j]))
    pairs = sorted(pairs, key=lambda x: x[1], reverse=True)[:TOP_K_PAIRS]
    for (a, b), val in pairs:
        plt.figure()
        plt.scatter(df_no_outliers[a], df_no_outliers[b], s=10)
        plt.title(f"Scatter — {a} vs {b} (|r|={val:.2f})")
        plt.xlabel(a); plt.ylabel(b)
        plt.tight_layout(); plt.show()
else:
    print("Not enough numeric features for pairwise scatter plots.")

In [ ]:
if TARGET_COLUMN is not None and TARGET_COLUMN in df_no_outliers.columns:
    y = df_no_outliers[TARGET_COLUMN]
    X = df_no_outliers.drop(columns=[TARGET_COLUMN])
    cats, nums = split_feature_types(df_no_outliers, target=TARGET_COLUMN)
    is_regression = pd.api.types.is_numeric_dtype(y) and y.nunique() > 15
    print(f"Target column: {TARGET_COLUMN} → {'Regression' if is_regression else 'Classification'} mode")    

    X_enc = X.copy()
    for c in cats:
        X_enc[c] = X_enc[c].astype("category").cat.codes
    X_enc = X_enc.fillna(X_enc.median(numeric_only=True))

    if is_regression:
        model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
        model.fit(X_enc, y)
        pred = model.predict(X_enc)
        print("MAE:", mean_absolute_error(y, pred))
        print("RMSE:", mean_squared_error(y, pred, squared=False))
        print("R2 :", r2_score(y, pred))
        importances = model.feature_importances_
    else:
        y_enc = y if pd.api.types.is_numeric_dtype(y) else LabelEncoder().fit_transform(y.astype(str))
        model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
        model.fit(X_enc, y_enc)
        pred = model.predict(X_enc)
        print("Accuracy:", accuracy_score(y_enc, pred))
        print("F1      :", f1_score(y_enc, pred, average='weighted'))
        importances = model.feature_importances_

    idx = np.argsort(importances)[-20:]
    names = X_enc.columns[idx]
    plt.figure()
    plt.barh(range(len(idx)), importances[idx])
    plt.yticks(range(len(idx)), names)
    plt.title("Feature importances (RandomForest)")
    plt.tight_layout(); plt.show()
else:
    print("TARGET_COLUMN not set or not found; skipping target analysis.")

In [ ]:
OUT_PATH = Path("artifacts") / "cleaned.csv"
OUT_PATH.parent.mkdir(exist_ok=True)
df_no_outliers.to_csv(OUT_PATH, index=False)
print("Saved cleaned dataset to:", OUT_PATH.resolve())